# Instacart Data Pipeline
## Stage 3: Gold (Dimensional Modeling + Validation)

**Owner:** Cath  
**Source schema:** `workspace.instacart_silver`  
**Target schema:** `workspace.instacart_gold`

### What this notebook does

Builds the Gold dimensional model from the cleaned Silver layer:

- `gold_dim_product` supplies product, aisle, and department context.
- `gold_dim_order` supplies customer, sequence, and time context.
- `gold_fact_order_product` records products appearing in orders.

It then runs three validation queries covering dimension quality,
intended keys and relationships, Silver-to-Gold row preservation, and
measure reconciliation.

### Required session context

The updated notebook does not contain a Gold setup cell. Its table names
are unqualified, so before Query 15 runs, the SQL session must already use:

```sql
USE CATALOG workspace;
USE SCHEMA instacart_gold;
```

The five Silver inputs must already exist under
`workspace.instacart_silver`.

### Build Order

| # | Query | Depends on | Why |
|---|---|---|---|
| 15 | `gold_dim_product` | Silver products, aisles, departments | Flattens the product hierarchy |
| 16 | `gold_dim_order` | Silver orders | Adds order day and interval labels |
| 17 | `gold_fact_order_product` | Silver order products | Creates the narrow product-line fact |
| 18 | Pre-constraint validation | Queries 15 and 16 | Reports dimension-key quality |
| 19 | Constraint checks | Queries 15–17 | Reports intended PK/FK data integrity |
| 20 | Final validation | Queries 15–17 | Reconciles Gold against Silver |

**Job DAG:** Queries 15, 16, and 17 can run in parallel after Silver
validation because they read separate Silver outputs. Query 18 waits for
both dimensions. Queries 19 and 20 wait for all three Gold tables.

### Design for scale

All three builds use `CREATE OR REPLACE TABLE ... AS SELECT`, so reruns
replace the snapshot rather than append duplicate copies. Product,
aisle, and department descriptions are flattened once in the product
dimension. The fact remains narrow and does not repeat those descriptions.

These are full-refresh builds and exact full-table checks. They suit this
homework snapshot but are not an incremental production design.

### Grain and key note

The business event is one product line in one labeled order.
This notebook declares and validates `(order_id, product_id)` as the
fact's composite key. The earlier agreed primary grain key was
`(order_id, add_to_cart_order)`, while `(order_id, product_id)` was an
additional uniqueness check. The supplied SQL does not validate the
basket-position key, and this documentation does not change it.

`add_to_cart_order` is basket position, not quantity.

## Part 1: Dimensions

Dimension tables contain descriptive context used to filter, group, and
label product-line events. Build both before relying on dimension-based
analytics.

### gold_dim_product

Joins `products_clean` to the cleaned aisle and department lookups.
The resulting table keeps one product identifier and adds readable
`aisle_name` and `department_name` attributes. Flattening this hierarchy
avoids additional aisle and department joins in dashboard queries.

The left joins retain products whose lookup row is missing, but their
names would be null. Unique Silver lookup keys are also required to
prevent the joins from multiplying product rows.

**Grain:** One row per product, identified by `product_id`.  
**Depends on:** `products_clean`, `aisles_clean`, and
`departments_clean`.  
**Expected result:** Gold product rows match Silver product rows when
lookup keys are unique.

In [0]:
%sql
-- Owner: Cath
-- Name: 15_gold_dim_product.sql
-- Purpose: Build a product dimension with denormalized aisle and department names.
-- Grain: One row per product, uniquely identified by product_id.

CREATE OR REPLACE TABLE gold_dim_product AS
SELECT
    p.product_id,
    p.product_name,
    p.aisle_id,
    a.aisle AS aisle_name,
    p.department_id,
    d.department AS department_name
FROM workspace.instacart_silver.products p
LEFT JOIN workspace.instacart_silver.aisles a ON p.aisle_id = a.aisle_id
LEFT JOIN workspace.instacart_silver.departments d ON p.department_id = d.department_id;

DESCRIBE TABLE gold_dim_product;

### gold_dim_order

Copies order and customer context from `orders_clean` and adds two
reader-friendly classifications:

- `order_day_name` maps codes 0–6 to Sunday–Saturday. That mapping is
  an assumption in this SQL and should be confirmed from approved source
  metadata before the names are presented as established facts.
- `order_frequency_category` groups the previous-order interval into
  First Order, Within 1 Week, 1–2 Weeks, 2–4 Weeks, or Over 30 Days.
  It describes this order's interval, not the customer's lifetime behavior.

The source interval remains available as `days_since_prior_order`.
Null is classified as First Order, relying on Silver validation to ensure
that only first orders have a null interval.

**Grain:** One row per order, identified by `order_id`.  
**Depends on:** `orders_clean`.  
**Expected result:** Gold order rows match all Silver order rows.

The final `SELECT *` is a data preview, not a validation check.

In [0]:
%sql
-- Owner: Cath
-- Name: 16_gold_dim_order.sql
-- Purpose: Build an order dimension with customer and time attributes.
-- Grain: One row per order, uniquely identified by order_id.

CREATE OR REPLACE TABLE gold_dim_order AS
SELECT
    order_id,
    user_id,
    order_number,
    order_dow,
    CASE order_dow
        WHEN 0 THEN 'Sunday'
        WHEN 1 THEN 'Monday'
        WHEN 2 THEN 'Tuesday'
        WHEN 3 THEN 'Wednesday'
        WHEN 4 THEN 'Thursday'
        WHEN 5 THEN 'Friday'
        WHEN 6 THEN 'Saturday'
    END AS order_day_name,
    order_hour_of_day,
    days_since_prior_order,
    CASE
        WHEN days_since_prior_order IS NULL THEN 'First Order'
        WHEN days_since_prior_order <= 7 THEN 'Within 1 Week'
        WHEN days_since_prior_order <= 14 THEN '1-2 Weeks'
        WHEN days_since_prior_order <= 30 THEN '2-4 Weeks'
        ELSE 'Over 30 Days'
    END AS order_frequency_category
FROM workspace.instacart_silver.orders;

DESCRIBE TABLE gold_dim_order;

## Part 2: Fact Table

### gold_fact_order_product

Creates the central, narrow fact by copying four fields from
`order_products_clean`:

| Column | Meaning |
|---|---|
| `order_id` | Intended reference to `gold_dim_order` |
| `product_id` | Intended reference to `gold_dim_product` |
| `add_to_cart_order` | Basket position |
| `reordered` | Silver boolean reorder indicator |

There are no joins, filters, or derived measures in this build. Therefore,
its row count should exactly match Silver order products. Descriptions
and order context are obtained by joining the fact to the dimensions.

**Grain declared by this cell:** One product line in one order, uniquely
identified by `(order_id, product_id)`.  
**Depends on:** `order_products_clean`.

Purchase count, reorder count, and reorder rate remain derived analytics:

- Purchase count: `COUNT(*)`
- Reorder count: count or sum of rows where `reordered = TRUE`
- Reorder rate: reordered rows divided by purchase rows

No quantity measure is created.


In [0]:
%sql
-- Owner: Cath
-- Name: 17_gold_fact_order_product.sql
-- Purpose: Build the fact table linking orders and products with behavioral metrics.
-- Grain: One row per product line in one order, uniquely identified by (order_id, product_id).

CREATE OR REPLACE TABLE gold_fact_order_product AS
SELECT
    op.order_id, -- connects to dim_order table
    op.product_id, -- connects to dim_product table
    op.add_to_cart_order,
    op.reordered
FROM workspace.instacart_silver.order_products op;

DESCRIBE TABLE gold_fact_order_product;

## Part 3: Validation

Three reports inspect Gold. Their `PASS` and `REVIEW` values are
informational: none of these queries uses `assert_true` to stop a Job.

### gold_pre_constraint_validation — Query 18

Reports one row for each dimension and checks:

- Null primary-key candidates.
- Duplicate identifier groups.
- Missing product names.
- Missing order `user_id` or `order_number`.

**Expected result:** Two `PASS` rows with zero issue counts.

The existing Purpose says this runs before the fact is built, but the
notebook places it after Query 17. It validates only the dimensions, so
the fact is not used by this query.

In [0]:
%sql
-- Owner: Cath
-- Name: 18_validate_gold_pre_constraints.sql
-- Purpose: Validate dimension tables before building the fact table.
-- Grain: One validation summary row per dimension table.

WITH validation AS (

    SELECT
        'gold_dim_product' AS table_name,
        COUNT(*) AS row_count,
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_key_rows,
        (SELECT COUNT(*) FROM (
            SELECT product_id FROM gold_dim_product
            WHERE product_id IS NOT NULL GROUP BY product_id HAVING COUNT(*) > 1
        )) AS duplicate_keys,
        SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END) AS required_field_issues
    FROM gold_dim_product

    UNION ALL

    SELECT
        'gold_dim_order',
        COUNT(*),
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END),
        (SELECT COUNT(*) FROM (
            SELECT order_id FROM gold_dim_order
            WHERE order_id IS NOT NULL GROUP BY order_id HAVING COUNT(*) > 1
        )),
        SUM(CASE WHEN user_id IS NULL OR order_number IS NULL THEN 1 ELSE 0 END)
    FROM gold_dim_order

)
SELECT
    table_name,
    row_count,
    null_key_rows,
    duplicate_keys,
    required_field_issues,
    CASE
        WHEN null_key_rows > 0 OR duplicate_keys > 0 OR required_field_issues > 0
        THEN 'REVIEW'
        ELSE 'PASS'
    END AS status
FROM validation
ORDER BY table_name;

### gold_constraint_checks — Query 19

Produces eight report rows for:

1. Product-key uniqueness.
2. Order-key uniqueness.
3. Fact `(order_id, product_id)` uniqueness.
4. Null product keys.
5. Null order keys.
6. Null fact order/product keys.
7. Fact-to-order relationship integrity.
8. Fact-to-product relationship integrity.

A row returns `PASS` when its violations equal zero; otherwise it returns
`REVIEW`.

Despite its filename and Purpose, this query only checks the data. It does
not execute `ALTER TABLE ... ADD CONSTRAINT`, register primary or foreign
keys, or create relationship lines in Catalog.

In [0]:
%sql
-- Owner: Cath
-- Name: 19_gold_constraints.sql
-- Purpose: Validate PK/FK constraints across all gold tables.
-- Grain: One validation summary row per constraint check.

WITH constraint_checks AS (

    -- Check 1: gold_dim_product PK uniqueness
    SELECT
        'PK: gold_dim_product.product_id' AS constraint_name,
        'Primary Key Uniqueness' AS constraint_type,
        (SELECT COUNT(*) FROM (
            SELECT product_id FROM gold_dim_product
            WHERE product_id IS NOT NULL
            GROUP BY product_id HAVING COUNT(*) > 1
        )) AS violations

    UNION ALL

    -- Check 2: gold_dim_order PK uniqueness
    SELECT
        'PK: gold_dim_order.order_id',
        'Primary Key Uniqueness',
        (SELECT COUNT(*) FROM (
            SELECT order_id FROM gold_dim_order
            WHERE order_id IS NOT NULL
            GROUP BY order_id HAVING COUNT(*) > 1
        ))

    UNION ALL

    -- Check 3: gold_fact_order_product composite PK uniqueness
    SELECT
        'PK: gold_fact_order_product (order_id, product_id)',
        'Composite Key Uniqueness',
        (SELECT COUNT(*) FROM (
            SELECT order_id, product_id FROM gold_fact_order_product
            WHERE order_id IS NOT NULL AND product_id IS NOT NULL
            GROUP BY order_id, product_id HAVING COUNT(*) > 1
        ))

    UNION ALL

    -- Check 4: gold_dim_product PK null check
    SELECT
        'PK: gold_dim_product.product_id NOT NULL',
        'Primary Key Null Check',
        (SELECT COUNT(*) FROM gold_dim_product WHERE product_id IS NULL)

    UNION ALL

    -- Check 5: gold_dim_order PK null check
    SELECT
        'PK: gold_dim_order.order_id NOT NULL',
        'Primary Key Null Check',
        (SELECT COUNT(*) FROM gold_dim_order WHERE order_id IS NULL)

    UNION ALL

    -- Check 6: gold_fact_order_product composite PK null check
    SELECT
        'PK: gold_fact_order_product (order_id, product_id) NOT NULL',
        'Composite Key Null Check',
        (SELECT COUNT(*) FROM gold_fact_order_product
         WHERE order_id IS NULL OR product_id IS NULL)

    UNION ALL

    -- Check 7: FK gold_fact_order_product.order_id → gold_dim_order.order_id
    SELECT
        'FK: gold_fact_order_product.order_id → gold_dim_order.order_id',
        'Foreign Key Integrity',
        (SELECT COUNT(*) FROM gold_fact_order_product f
         LEFT JOIN gold_dim_order d ON f.order_id = d.order_id
         WHERE f.order_id IS NOT NULL AND d.order_id IS NULL)

    UNION ALL

    -- Check 8: FK gold_fact_order_product.product_id → gold_dim_product.product_id
    SELECT
        'FK: gold_fact_order_product.product_id → gold_dim_product.product_id',
        'Foreign Key Integrity',
        (SELECT COUNT(*) FROM gold_fact_order_product f
         LEFT JOIN gold_dim_product d ON f.product_id = d.product_id
         WHERE f.product_id IS NOT NULL AND d.product_id IS NULL)

)
SELECT
    constraint_name,
    constraint_type,
    violations,
    CASE WHEN violations = 0 THEN 'PASS' ELSE 'REVIEW' END AS status
FROM constraint_checks
ORDER BY constraint_type, constraint_name;

### gold_final_validation — Query 20

Returns two result sets. Both must be reviewed.

#### Result set 1 — Table-level checks

One row for each Gold table, showing its Gold count, Silver source count,
row difference, null keys, duplicate-key groups, required-field issues,
and unmatched fact relationships.

A Gold row-count difference causes `REVIEW` because these builds are
intended to preserve their Silver source rows.

**Expected result:** Three `PASS` rows with zero differences and issues.

#### Result set 2 — Measure reconciliation

Compares these values between Silver order products and the Gold fact:

| Measure | Meaning |
|---|---|
| `total_order_lines` | Total product-line purchase events |
| `distinct_orders` | Orders represented in the fact |
| `distinct_products` | Products represented in the fact |
| `reordered_count` | Rows where `reordered = TRUE` |
| `total_cart_position_sum` | Checksum of stored basket positions |

The final difference is `silver_value - gold_value`; all five values
should be zero and return `PASS`.

The cart-position sum is a reconciliation checksum, not purchase quantity.
Matching aggregate totals also do not prove every individual row matches.

#### Current coverage limits

- The reports do not stop execution with `assert_true`.
- They check `(order_id, product_id)`, not the agreed
  `(order_id, add_to_cart_order)` grain key.
- They do not explicitly check null/invalid basket positions or validate
  that `reordered` contains only valid values.
- Query 19 reports intended constraints but does not register them.
- The dimension reports do not prove aisle/department names are complete
  or that the day-name mapping is correct.

These notes describe the supplied SQL; they are not implemented changes.

In [0]:
%sql
-- Owner: Cath
-- Name: 20_validate_gold_final.sql
-- Purpose: Comprehensive gold validation - keys, integrity, silver-to-gold reconciliation, and measures.
-- Grain: Two result sets - (1) table-level validation, (2) measure reconciliation.

-- PART 1: Table-level validation (keys, row counts, referential integrity)
WITH validation AS (

    SELECT
        'gold_dim_product' AS table_name,
        COUNT(*) AS row_count,
        (SELECT COUNT(*) FROM workspace.instacart_silver.products) AS source_row_count,
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_key_rows,
        (SELECT COUNT(*) FROM (
            SELECT product_id FROM gold_dim_product
            WHERE product_id IS NOT NULL GROUP BY product_id HAVING COUNT(*) > 1
        )) AS duplicate_keys,
        SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END) AS required_field_issues,
        0 AS unmatched_fk_rows
    FROM gold_dim_product

    UNION ALL

    SELECT
        'gold_dim_order',
        COUNT(*),
        (SELECT COUNT(*) FROM workspace.instacart_silver.orders),
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END),
        (SELECT COUNT(*) FROM (
            SELECT order_id FROM gold_dim_order
            WHERE order_id IS NOT NULL GROUP BY order_id HAVING COUNT(*) > 1
        )),
        SUM(CASE WHEN user_id IS NULL OR order_number IS NULL THEN 1 ELSE 0 END),
        0
    FROM gold_dim_order

    UNION ALL

    SELECT
        'gold_fact_order_product',
        COUNT(*),
        (SELECT COUNT(*) FROM workspace.instacart_silver.order_products),
        SUM(CASE WHEN order_id IS NULL OR product_id IS NULL THEN 1 ELSE 0 END),
        (SELECT COUNT(*) FROM (
            SELECT order_id, product_id FROM gold_fact_order_product
            WHERE order_id IS NOT NULL AND product_id IS NOT NULL
            GROUP BY order_id, product_id HAVING COUNT(*) > 1
        )),
        0,
        -- Check referential integrity: fact → dimensions
        (SELECT COUNT(*) FROM gold_fact_order_product f
            LEFT JOIN gold_dim_order o ON f.order_id = o.order_id
            WHERE f.order_id IS NOT NULL AND o.order_id IS NULL)
         + (SELECT COUNT(*) FROM gold_fact_order_product f
            LEFT JOIN gold_dim_product p ON f.product_id = p.product_id
            WHERE f.product_id IS NOT NULL AND p.product_id IS NULL)
    FROM gold_fact_order_product

)
SELECT
    table_name,
    row_count,
    source_row_count,
    row_count - source_row_count AS row_difference,
    null_key_rows,
    duplicate_keys,
    required_field_issues,
    unmatched_fk_rows,
    CASE
        WHEN null_key_rows > 0 OR duplicate_keys > 0
          OR required_field_issues > 0 OR unmatched_fk_rows > 0
          OR row_count <> source_row_count
        THEN 'REVIEW'
        ELSE 'PASS'
    END AS status
FROM validation
ORDER BY table_name;

-- PART 2: Measure reconciliation (aggregate validation)
WITH silver_measures AS (
    SELECT
        COUNT(*) AS total_order_lines,
        COUNT(DISTINCT order_id) AS distinct_orders,
        COUNT(DISTINCT product_id) AS distinct_products,
        SUM(CASE WHEN reordered = 1 THEN 1 ELSE 0 END) AS reordered_count,
        SUM(add_to_cart_order) AS total_cart_position_sum
    FROM workspace.instacart_silver.order_products
),
gold_measures AS (
    SELECT
        COUNT(*) AS total_order_lines,
        COUNT(DISTINCT order_id) AS distinct_orders,
        COUNT(DISTINCT product_id) AS distinct_products,
        SUM(CASE WHEN reordered = 1 THEN 1 ELSE 0 END) AS reordered_count,
        SUM(add_to_cart_order) AS total_cart_position_sum
    FROM gold_fact_order_product
)
SELECT
    'total_order_lines' AS measure,
    s.total_order_lines AS silver_value,
    g.total_order_lines AS gold_value,
    s.total_order_lines - g.total_order_lines AS difference,
    CASE WHEN s.total_order_lines = g.total_order_lines THEN 'PASS' ELSE 'REVIEW' END AS status
FROM silver_measures s, gold_measures g

UNION ALL

SELECT
    'distinct_orders',
    s.distinct_orders,
    g.distinct_orders,
    s.distinct_orders - g.distinct_orders,
    CASE WHEN s.distinct_orders = g.distinct_orders THEN 'PASS' ELSE 'REVIEW' END
FROM silver_measures s, gold_measures g

UNION ALL

SELECT
    'distinct_products',
    s.distinct_products,
    g.distinct_products,
    s.distinct_products - g.distinct_products,
    CASE WHEN s.distinct_products = g.distinct_products THEN 'PASS' ELSE 'REVIEW' END
FROM silver_measures s, gold_measures g

UNION ALL

SELECT
    'reordered_count',
    s.reordered_count,
    g.reordered_count,
    s.reordered_count - g.reordered_count,
    CASE WHEN s.reordered_count = g.reordered_count THEN 'PASS' ELSE 'REVIEW' END
FROM silver_measures s, gold_measures g

UNION ALL

SELECT
    'total_cart_position_sum',
    s.total_cart_position_sum,
    g.total_cart_position_sum,
    s.total_cart_position_sum - g.total_cart_position_sum,
    CASE WHEN s.total_cart_position_sum = g.total_cart_position_sum THEN 'PASS' ELSE 'REVIEW' END
FROM silver_measures s, gold_measures g;

## Gold Layer: Summary

| Gold table | Validation target | Role |
|---|---|---|
| `gold_dim_product` | PASS | Product dimension with flattened aisle and department descriptions |
| `gold_dim_order` | PASS | Order dimension with customer, time, and interval context |
| `gold_fact_order_product` | PASS | Narrow product-line fact |

The final model contains one fact table, not two. The fact's `product_id`
is intended to refer to `gold_dim_product`, and its `order_id` is
intended to refer to `gold_dim_order`.

**Expected reports:**

- Query 18: 2 PASS rows.
- Query 19: 8 PASS rows with zero violations.
- Query 20 result set 1: 3 PASS rows with zero differences/issues.
- Query 20 result set 2: 5 PASS rows with zero differences.

These are expected targets, not recorded evidence that the notebook ran.

**Next stage:** Analytics and dashboards should use the actual final Gold
table and column names. Product context comes from `gold_dim_product`;
customer and time context comes from `gold_dim_order`. A products-bought-
together analysis can self-join the fact by `order_id` without changing
the fact's product-line grain.

*This documented copy preserves all six original SQL cells in their
original order. Only Markdown documentation and the notebook display name
were added. No SQL was executed.*
